# Heart Disease — XGBoost, LightGBM & SHAP
## Project 10

Compare **XGBoost** and **LightGBM** for binary classification and interpret XGBoost
with **SHAP** and permutation importance.

> Dataset: reproducible educational practice data; not a clinical dataset.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split,StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,log_loss,confusion_matrix,RocCurveDisplay
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import shap
BASE=Path.cwd()
if not (BASE/'dataset').exists(): BASE=BASE.parent
DATA=BASE/'dataset'/'heart_disease.csv'; OUT=BASE/'outputs'; OUT.mkdir(exist_ok=True)
df=pd.read_csv(DATA)
display(df.head()); print(df.shape)

## 1. Data Quality and EDA

In [ ]:
display(df.describe().T)
display(pd.DataFrame({'Missing_Count':df.isna().sum(),'Missing_Percent':df.isna().mean()*100}))
display(df.target.value_counts().sort_index())

In [ ]:
plt.figure(figsize=(7,5)); plt.hist(df.age,bins=20); plt.title('Age Distribution'); plt.xlabel('Age'); plt.ylabel('Frequency'); plt.show()
plt.figure(figsize=(7,5)); df.groupby('cp').target.mean().plot(kind='bar'); plt.title('Disease Rate by Chest Pain Category'); plt.show()

## 2. Preprocessing

In [ ]:
num=['age','trestbps','chol','thalach','oldpeak']
cat=['sex','cp','fbs','restecg','exang','slope','ca','thal']
X=df[num+cat]; y=df.target
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,stratify=y,random_state=42)
pre=ColumnTransformer([
 ('num',SimpleImputer(strategy='median'),num),
 ('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),
                  ('onehot',OneHotEncoder(handle_unknown='ignore',drop='first'))]),cat)
])

## 3. XGBoost

In [ ]:
xgb=XGBClassifier(n_estimators=160,max_depth=4,learning_rate=.05,subsample=.85,
 colsample_bytree=.85,reg_lambda=1,random_state=42,n_jobs=1,eval_metric='logloss')
xgb_pipe=Pipeline([('preprocessor',pre),('model',xgb)])
xgb_pipe.fit(X_train,y_train)
xgb_pred=xgb_pipe.predict(X_test); xgb_prob=xgb_pipe.predict_proba(X_test)[:,1]

## 4. LightGBM

In [ ]:
lgbm=LGBMClassifier(n_estimators=160,max_depth=5,num_leaves=20,learning_rate=.05,
 subsample=.85,colsample_bytree=.85,reg_lambda=1,random_state=42,n_jobs=1,verbosity=-1)
lgbm_pipe=Pipeline([('preprocessor',pre),('model',lgbm)])
lgbm_pipe.fit(X_train,y_train)
lgb_pred=lgbm_pipe.predict(X_test); lgb_prob=lgbm_pipe.predict_proba(X_test)[:,1]

## 5. Model Comparison

In [ ]:
comparison=pd.DataFrame([
 {'Model':'XGBoost','Accuracy':accuracy_score(y_test,xgb_pred),'Precision':precision_score(y_test,xgb_pred),'Recall':recall_score(y_test,xgb_pred),'F1':f1_score(y_test,xgb_pred),'ROC_AUC':roc_auc_score(y_test,xgb_prob),'Log_Loss':log_loss(y_test,xgb_prob)},
 {'Model':'LightGBM','Accuracy':accuracy_score(y_test,lgb_pred),'Precision':precision_score(y_test,lgb_pred),'Recall':recall_score(y_test,lgb_pred),'F1':f1_score(y_test,lgb_pred),'ROC_AUC':roc_auc_score(y_test,lgb_prob),'Log_Loss':log_loss(y_test,lgb_prob)}
])
display(comparison); comparison.to_csv(OUT/'model_comparison_metrics.csv',index=False)

## 6. ROC Curves

In [ ]:
RocCurveDisplay.from_predictions(y_test,xgb_prob,name='XGBoost')
RocCurveDisplay.from_predictions(y_test,lgb_prob,name='LightGBM')
plt.title('XGBoost vs LightGBM ROC Curves'); plt.show()

## 7. SHAP Explainability

In [ ]:
transformer=xgb_pipe.named_steps['preprocessor']
core=xgb_pipe.named_steps['model']
Xt=transformer.transform(X_test)
names=transformer.get_feature_names_out()
explainer=shap.TreeExplainer(core)
sv=explainer.shap_values(Xt)
if isinstance(sv,list): sv=sv[1]
shap_importance=pd.DataFrame({'Feature':names,'Mean_Abs_SHAP':np.abs(sv).mean(axis=0)}).sort_values('Mean_Abs_SHAP',ascending=False)
display(shap_importance.head(20))
shap_importance.to_csv(OUT/'shap_feature_importance.csv',index=False)

## 8. Permutation Importance

In [ ]:
perm=permutation_importance(xgb_pipe,X_test,y_test,n_repeats=5,random_state=42,scoring='roc_auc',n_jobs=1)
perm_df=pd.DataFrame({'Feature':X.columns,'Importance_Mean':perm.importances_mean,'Importance_Std':perm.importances_std}).sort_values('Importance_Mean',ascending=False)
display(perm_df); perm_df.to_csv(OUT/'permutation_importance.csv',index=False)

## 9. Five-Fold Cross-Validation

In [ ]:
cv=StratifiedKFold(5,shuffle=True,random_state=42)
rows=[]
for name,est in [('XGBoost',xgb),('LightGBM',lgbm)]:
    vals={'Accuracy':[],'F1':[],'ROC_AUC':[]}
    for tr,va in cv.split(X,y):
        pipe=Pipeline([('preprocessor',pre),('model',est)])
        pipe.fit(X.iloc[tr],y.iloc[tr])
        pp=pipe.predict(X.iloc[va]); qq=pipe.predict_proba(X.iloc[va])[:,1]
        vals['Accuracy'].append(accuracy_score(y.iloc[va],pp)); vals['F1'].append(f1_score(y.iloc[va],pp)); vals['ROC_AUC'].append(roc_auc_score(y.iloc[va],qq))
    rows.append({'Model':name,'Accuracy_Mean':np.mean(vals['Accuracy']),'F1_Mean':np.mean(vals['F1']),'ROC_AUC_Mean':np.mean(vals['ROC_AUC'])})
cv_summary=pd.DataFrame(rows); display(cv_summary); cv_summary.to_csv(OUT/'cross_validation_comparison.csv',index=False)

## 10. Conclusion

**Data → preprocessing → XGBoost + LightGBM → model comparison → ROC-AUC/F1 →
cross-validation → permutation importance → SHAP explainability.**

The project demonstrates advanced tree-based boosting and model interpretation.
It is intended for educational machine-learning practice only.